##### ***Let's build a Simple RAG Application***

#### ***What is RAG?***

#### ***RAG(Retrieval Augmented Generation) is a technique where an LLM retrieves relevant dovcuments from the external source , use that information to generate an answer***

In [19]:
#### Load API Key

from dotenv import load_dotenv

load_dotenv()

True

In [20]:
#### Load the webpage

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Rishabh_Pant")

documents = loader.load()


In [21]:
print("Number Of Documents:",len(documents))

Number Of Documents: 1


In [22]:
documents

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Rishabh_Pant', 'title': 'Rishabh Pant - Wikipedia', 'language': 'en'}, page_content='\n\n\n\nRishabh Pant - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nEarly life\n\n\n\n\n\n\n\n\n2\nDomestic career\n\n\n\n\nToggle

In [23]:
### Create a chunks using RecursiveCharacterTextSplitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)
print("Number Of Chunks:",len(chunks))

Number Of Chunks: 59


In [24]:
for chunk in chunks:
    print(len(chunk.page_content))

783
737
435
511
773
469
701
395
448
669
589
487
412
438
789
622
444
748
627
757
785
684
778
629
782
125
687
633
791
796
664
737
776
778
782
727
763
767
747
762
684
794
766
678
639
773
676
694
767
590
83
795
783
632
696
89
785
208
676


In [25]:
### Create Embedding Model

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4387.37it/s]


In [26]:
#create a embedding vector for each chunk 
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

In [27]:
### Retriever with vector semantic search
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":5}
)

In [28]:

### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000147617B3AA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001470AF58E60>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [29]:
### Design a simple prompt

from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("""
        You are an Helpful Ai Assistant.

        Please answer the question based on provided context only.

        if you don't find the answer in provided context, say I don't find the answer on provided context.

        Context:{context}

        Question:{question}
""")

In [30]:
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

def format_docs(docs):

    return "\n\n".join(doc.page_content for doc in docs)

final_chain = (
    {
        "context":retriever | RunnableLambda(format_docs),
        "question":RunnablePassthrough()
    }
    | prompt_template 
    | llm
    | output_parser
)

In [31]:
question = "When was his first hundered in Test against which team?"

In [32]:
response = final_chain.invoke(question)
print(response)

His first Test hundred came on **11 September 2018**, and it was scored **against England**.
